Breakdown into steps:

1. Find the data (informed by the snakemake output paths)
2. Define data parses (JSON easy, but for logs in plain text may be harder)
3. Quantify the data and collect into a normalized table, indexed on sample
4. Visualize with datavzrd (pretty table)
5. Add plots where necessary.

This gives full freedom to incorporate custom QC data into the report! No more fuddling with the MultiQC config files 😭

After staring at the datavzrd examples, for a bit, here is what I want to do in no certain order:

- use YTE for sure to make config file definition more consistent/systematic
- definitely use the link-to-url functionality (template url could be some path to https://users.wenglab.org/ramirezc/{value})
  - link out to plots in matplotlib
- For summary plots, explore using the *.vl.json (this is a whole other rabbit hole)
- The pills plot are uniquely useull for caegorical data with multiple values per category (delimited by string). Think checkbox data.
- Custom functions, though this should be done in python first (really only useful for small conversions)
- How the whole link thing works on the entry level:

```yaml
link to oscar entry:
    column: Title
    table-row: oscars/movie
```

This is a link from the movies dataset to the oscars dataset. The key is the Title. So move with {Title} is matched to row in oscars table where the movie column is {Title}.

- Wait... This is going to be EXTREMELY useful for navigating QC metrics between different sequencing types! For now though focus on just the one.
- Ellipsis to cutoff long strings
- Regex to match columns with a certain prefix/pattern

In [ ]:
import os
import glob
import numpy as np
import re
import json
import yaml
from tqdm.auto import tqdm
from yte import process_yaml
import shlex
import subprocess
import pysam
import pandas as pd
import shutil
from collections import defaultdict, OrderedDict

os.chdir("/zata/zippy/ramirezc/atac-smk")

In [ ]:
samples = [f"EA1{i:05d}" for i in range(1, 292)]
sample_to_files = OrderedDict({sample: [] for sample in samples}) # initalized by samples wildcards but for now we just use glob

# del sample_to_files["EA100015"] # copy error

In [ ]:
glob_pattern = "*.json"
regex_pattern = r"(.*).json"

fastp_pattern = os.path.join("results", "fastp", glob_pattern)

sample_pattern = re.compile(os.path.join("results", "fastp", regex_pattern))

fastp_files = glob.glob(fastp_pattern)

for f in fastp_files:
    sample_to_files[sample_pattern.match(f).group(1)].append(f)

In [ ]:
glob_pattern = "*.narrowPeak"
regex_pattern = r"(.*).GRCH38_peaks.narrowPeak"

macs3_pattern = os.path.join("results", "macs3_callpeak", "0.05", glob_pattern)
 
sample_pattern = re.compile(os.path.join("results", "macs3_callpeak", "0.05", regex_pattern))

macs3_files = glob.glob(macs3_pattern)

for f in macs3_files:
    sample_to_files[sample_pattern.match(f).group(1)].append(f)
MACS3_FRAG_LEN = 150 # Informed by --extsize 150 in macs3_callpeak rule (so make a config parameter and pass to this script/notebook)

In [ ]:
glob_pattern = "*.json"
regex_pattern = r"(.*)-GRCH38.flagstat.json"

align_bam = os.path.join("results", "bowtie2_align", glob_pattern)
filter_bam = os.path.join("results", "filter", glob_pattern)
# picard_bam = os.path.join("results", "picard", glob_pattern)

align_pattern = re.compile(os.path.join("results", "bowtie2_align", regex_pattern))
filter_pattern = re.compile(os.path.join("results", "filter", regex_pattern))
# picard_pattern = re.compile(os.path.join("results", "picard", regex_pattern))

align_json_files = glob.glob(align_bam)
filter_json_files = glob.glob(filter_bam)
# picard_json_files = glob.glob(picard_bam)

for f in align_json_files:
    sample_to_files[align_pattern.match(f).group(1)].append(f)
for f in filter_json_files:
    sample_to_files[filter_pattern.match(f).group(1)].append(f)
# for f in picard_json_files:
    # sample_to_files[picard_pattern.match(f).group(1)].append(f)

In [ ]:
glob_pattern = "*.pbc.json"
regex_pattern = r"(.*)-GRCH38.pbc.json"

pbc_json_pattern = os.path.join("results", "pbc_qc", glob_pattern)
 
sample_pattern = re.compile(os.path.join("results", "pbc_qc", regex_pattern))

pbc_json_files = glob.glob(pbc_json_pattern)

for f in pbc_json_files:
    sample_to_files[sample_pattern.match(f).group(1)].append(f)

In [ ]:
glob_pattern = "*.txt"
regex_pattern = r"(.*)-GRCH38.txt"

tss_enrich_pattern = os.path.join("results", "tss_enrichment", glob_pattern)
 
sample_pattern = re.compile(os.path.join("results", "tss_enrichment", regex_pattern))

tss_enrich_files = glob.glob(tss_enrich_pattern)

print(tss_enrich_files[:1])

for f in tss_enrich_files:
    sample_to_files[sample_pattern.match(f).group(1)].append(f)

In [ ]:
glob_pattern = "*.txt"
regex_pattern = r"(.*)-GRCH38.txt"

frip_pattern = os.path.join("results", "frip_all", "0.05", glob_pattern)
 
sample_pattern = re.compile(os.path.join("results", "frip_all", "0.05", regex_pattern))

frip_files = glob.glob(frip_pattern)

for f in frip_files:
    sample_to_files[sample_pattern.match(f).group(1)].append(f)

In [ ]:
molecular_df = pd.read_csv("/zata/zippy/ramirezc/mohd-api/data/Molecular-Data-Clean.tsv", sep="\t", header=0, usecols=["site", '{"ATAC","BC"}', '{"ATAC","PC"}', '{"ATAC","PO"}'])
display(molecular_df)

bc_df = molecular_df[['site', '{"ATAC","BC"}']].dropna()
bc_dict = dict(zip(bc_df['{"ATAC","BC"}'], bc_df.site))
pc_df = molecular_df[['site', '{"ATAC","PC"}']].dropna()
pc_dict = dict(zip(pc_df['{"ATAC","PC"}'], pc_df.site))
po_df = molecular_df[['site', '{"ATAC","PO"}']].dropna()
po_dict = dict(zip(po_df['{"ATAC","PO"}'], po_df.site))

acc_to_site = bc_dict | pc_dict | po_dict

print(list(acc_to_site.items())[:20])

In [ ]:
# glob_pattern = "*.bam"
# regex_pattern = r"(.*)-GRCH38.bam"

# align_bam = os.path.join("results", "bowtie2_align", glob_pattern)
# filter_bam = os.path.join("results", "filter", glob_pattern)
# picard_bam = os.path.join("results", "picard", glob_pattern)

# align_pattern = re.compile(os.path.join("results", "bowtie2_align", regex_pattern))
# filter_pattern = re.compile(os.path.join("results", "filter", regex_pattern))
# picard_pattern = re.compile(os.path.join("results", "picard", regex_pattern))

# align_bam_files = glob.glob(align_bam)
# filter_bam_files = glob.glob(filter_bam)
# picard_bam_files = glob.glob(picard_bam)

# for f in align_bam_files:
#     sample_to_files[align_pattern.match(f).group(1)].append(f)
# for f in filter_bam_files:
#     sample_to_files[filter_pattern.match(f).group(1)].append(f)
# for f in picard_bam_files:
#     sample_to_files[picard_pattern.match(f).group(1)].append(f)

In [ ]:
# from concurrent.futures import ProcessPoolExecutor

# def writeFlagstatJSON(args):
#     align_bam_file, deduped_bam_file, picard_bam_file = args
#     with open(f"{os.path.splitext(align_bam_file)[0]}.flagstat.json", "wb") as align_json_file, open(f"{os.path.splitext(deduped_bam_file)[0]}.flagstat.json", "wb") as deduped_json_file, open(f"{os.path.splitext(picard_bam_file)[0]}.flagstat.json", "wb") as picard_json_file:
#         align_json = subprocess.check_output(shlex.split(f"samtools flagstat --output-fmt json {align_bam_file}"))
#         deduped_json = subprocess.check_output(shlex.split(f"samtools flagstat --output-fmt json {deduped_bam_file}"))
#         picard_json = subprocess.check_output(shlex.split(f"samtools flagstat --output-fmt json {picard_bam_file}"))
        
#         align_json_file.write(align_json)
#         deduped_json_file.write(deduped_json)
#         picard_json_file.write(picard_json)
        
#     print(f"Wrote flagstat JSON files for {align_bam_file}, {deduped_bam_file}, and {picard_bam_file}")

# args = [(align_bam_file, deduped_bam_file, picard_bam_file) for (_, (_, _, align_bam_file, deduped_bam_file, picard_bam_file)) in sample_to_files.items()]
# with ProcessPoolExecutor(max_workers=190) as executor:
#     futures = []
#     for arg in args:
#         future = executor.submit(writeFlagstatJSON, arg)
#         futures.append(future)
#     for future in tqdm(futures):
#         future.result()

In [ ]:
# items = {}
# regex_pattern = re.compile(r"(.*)-GRCH38.png")
# for f in os.listdir("results/tss_enrichment"):
#     sample = regex_pattern.match(f)
#     if sample is None:
#         continue
#     sample = sample.group(1)
#     items[sample] = f"results/tss_enrichment/{f}"

# sorted_samples = sorted(items.keys())

# outdir = "/zata/public_html/users/ramirezc/mohd/atac/tss_enrichment"
# os.makedirs(outdir, exist_ok=True)
# for sample in sorted_samples:
#     # print(f"Copying {items[sample]} to {outdir}/{sample}.png")
#     shutil.copyfile(items[sample], os.path.join(outdir, f"{sample}.png"))

In [ ]:
fieldnames = [
    "Sample ID", 
    "Site",
    "Reads Before Filtering", 
    "Reads After Filtering", 
    r"% Filtered Reads",
    r"% of Reads >= MAPQ of 30", 
    "Number of Bases >= MAPQ of 30", 
    "GC Content", 
    "Mean Read 1 Length",
    "Mean Read 2 Length", 
    r"% Duplication",
    "Number of Peaks",
    "Fragment Length",
    "Reads Mapped (bowtie2)",
    "% Reads Mapped (bowtie2)",
    "Reads Mapped (samtools filter & deduped)",
    # "Reads Mapped (picard)",
    "PBC1",
    "PBC2",
    "NRF",
    "TSS Enrichment Score",
    "FRiP Score"
]
dtypes = [
    'U8', 
    'U3',
    int, 
    int, 
    float, 
    float, 
    int, 
    float, 
    int, 
    int, 
    float,
    int,
    int,
    int,
    float,
    int,
    # int,
    float,
    float,
    float,
    float,
    float
]
fmt_specifier_list = ['%s', '%s', '%d', '%d', '%.4f', '%.4f', '%d', '%.4f', '%d', '%d', '%.4f', '%d', '%d', '%d', '%.4f', '%d', '%.4f', '%.4f', '%.4f', '%.4f', '%.4f']
assert len(fieldnames) == len(dtypes) == len(fmt_specifier_list), "Lengths of fieldnames, dtypes, and fmt_specifier_list must be equal"

In [ ]:
def _make_gen(reader):
    b = reader(1024 * 1024)
    while b:
        yield b
        b = reader(1024*1024)

def rawgencount(filename):
    f = open(filename, 'rb')
    f_gen = _make_gen(f.raw.read)
    return sum(buf.count(b'\n') for buf in f_gen)

def getReadCount(bamFile, getMapped=True):
    "Get the number of mapped reads in a BAM file. Specifically excludes reads from chrM and chrUn_*."
    with pysam.AlignmentFile(bamFile) as bam:
        assert bam.has_index(), "BAM file must have an index to perform this operation. Run `samtools index` on the BAM file first."
        idxstats = np.fromiter(((stat.contig, stat.mapped, stat.unmapped, stat.total) for stat in bam.get_index_statistics()), dtype=[('seq', 'U30'), ('mapped', int), ('unmapped', int), ('total', float)])
        mask = (idxstats['seq'] != 'chrM') & (~(np.char.startswith(idxstats['seq'], 'chrUn_')))
        count = np.sum(idxstats['mapped'][mask])
        if not getMapped:
            count = np.sum(idxstats['unmapped'][mask])
        return int(count)

In [ ]:
print(list(sample_to_files.items())[:16])

In [35]:
height = len(fastp_files)

struct_dtype = np.dtype(list(zip(fieldnames, dtypes)))
records = np.empty(height, dtype=struct_dtype)

for i, (sample, [fastp_file, macs3_file, align_json_file, filter_json_file, pbc_json_file, tss_enrich_file, frip_file]) in enumerate(sample_to_files.items()):
    with open(fastp_file) as f:
        fastp_json = json.load(f)
    
    record = records[i]
    
    if sample == "EA100015" or sample == "EA100185":
        continue
    
    is_paired_end = True if fastp_json["summary"]["sequencing"].startswith("paired end") else False
    reads_before_filtering = fastp_json["summary"]["before_filtering"]["total_reads"]
    reads_after_filtering = fastp_json["summary"]["after_filtering"]["total_reads"]
    passed_filter_reads = fastp_json["filtering_result"]["passed_filter_reads"]
    pct_filter_reads = passed_filter_reads / reads_before_filtering
    pct_gt_q30 = fastp_json["summary"]["after_filtering"]["q30_rate"] # Q30 meaning: https://www.illumina.com/documents/products/technotes/technote_understanding_quality_scores.pdf
    no_bases_q30 = fastp_json["summary"]["after_filtering"]["q30_bases"]
    gc_content = fastp_json["summary"]["after_filtering"]["gc_content"]
    mean_r1_length = fastp_json["summary"]["after_filtering"]["read1_mean_length"]
    mean_r2_length = fastp_json["summary"]["after_filtering"]["read2_mean_length"]
    pct_duplication = fastp_json["duplication"]["rate"]
    
    if is_paired_end:
        reads_before_filtering = reads_before_filtering // 2
        reads_after_filtering = reads_after_filtering // 2
        passed_filter_reads = passed_filter_reads // 2
    
    record["Sample ID"] = sample
    record["Site"] = acc_to_site[sample]
    record["Reads Before Filtering"] = reads_before_filtering
    record["Reads After Filtering"] = reads_after_filtering
    record[r"% Filtered Reads"] = pct_filter_reads
    record[r"% of Reads >= MAPQ of 30"] = pct_gt_q30
    record["Number of Bases >= MAPQ of 30"] = no_bases_q30
    record["GC Content"] = gc_content
    record["Mean Read 1 Length"] = mean_r1_length
    record["Mean Read 2 Length"] = mean_r2_length
    record[r"% Duplication"] = pct_duplication

    no_of_peaks = rawgencount(macs3_file)
    fragment_length = MACS3_FRAG_LEN
    
    record["Number of Peaks"] = no_of_peaks
    record["Fragment Length"] = fragment_length
    
    with open(align_json_file) as align_f, open(filter_json_file) as filter_f:
        align_json = json.load(align_f)
        filter_json = json.load(filter_f)
        # picard_json = json.load(picard_f)
    
    align_mapped_reads = align_json["QC-passed reads"]["primary mapped"]
    filter_mapped_reads = filter_json["QC-passed reads"]["primary mapped"]
    # picard_mapped_reads = picard_json["QC-passed reads"]["mapped"]
    
    if is_paired_end:
        align_mapped_reads = align_mapped_reads // 2
        filter_mapped_reads = filter_mapped_reads // 2
        # picard_mapped_reads = picard_mapped_reads // 2
    
    align_mapped_pct = align_json["QC-passed reads"]["primary mapped %"] / 100
    
    record['Reads Mapped (bowtie2)'] = align_mapped_reads
    record['% Reads Mapped (bowtie2)'] = align_mapped_pct
    record['Reads Mapped (samtools filter & deduped)'] = filter_mapped_reads
    # record['Reads Mapped (picard)'] = picard_mapped_reads
    
    with open(pbc_json_file) as pbc_f:
        pbc_json = json.load(pbc_f)
    
    PBC1 = pbc_json["PBC1"]
    PBC2 = pbc_json["PBC2"]
    NRF = pbc_json["NRF"]
    
    record["PBC1"] = PBC1
    record["PBC2"] = PBC2
    record["NRF"] = NRF
    
    with open(tss_enrich_file) as tss_f:
        tss_json = json.load(tss_f)
    
    record["TSS Enrichment Score"] = tss_json["tss_enrichment_score"]
    
    with open(frip_file) as frip_f:
        frip = float(frip_f.read().lstrip("Fraction of reads in peaks: ").strip())
    
    record["FRiP Score"] = frip
    
np.savetxt("qc_metrics.csv", records[np.where(records["Site"] != "")], delimiter=",", fmt=fmt_specifier_list, header=",".join(fieldnames))

In [36]:
# with open("datavzrd.config.yte.yaml", "w+") as template, open("datavzrd.config.yaml", "w") as yml:
#     template.write(datavzrd_yaml)
#     template.seek(0)
#     render = process_yaml(template, variables=variables)
#     yaml.dump(render, yml, default_flow_style=False, sort_keys=False)

In [43]:
subprocess.check_call(shlex.split("datavzrd datavzrd.config.yaml --output example-report --overwrite-output"))

0

In [ ]:
debug = False

for sample in samples:
    if sample == "EA100015" or sample == "EA100185":
        continue
    tss_dest_template = "/zata/public_html/users/ramirezc/mohd/atac/tss_enrichment/{value}.png"
    frag_len_dest_template = "/zata/public_html/users/ramirezc/mohd/atac/frag_len/{value}.png"
    tss_src_template = "/zata/zippy/ramirezc/atac-smk/results/tss_enrichment/{value}-GRCH38.png"
    frag_len_src_template = "/zata/zippy/ramirezc/atac-smk/results/frag_len/{value}-GRCH38.png"
    
    tss_dest = tss_dest_template.format(value=sample)
    frag_len_dest = frag_len_dest_template.format(value=sample)
    tss_src = tss_src_template.format(value=sample)
    frag_len_src = frag_len_src_template.format(value=sample)

    if not os.path.exists(os.path.dirname(tss_dest)) or not os.path.exists(os.path.dirname(frag_len_dest)):
        print(f"Destination does not exist for {sample}")
        break

    if debug:
        print(f"Copying {tss_src} to {tss_dest}")
        print(f"Copying {frag_len_src} to {frag_len_dest}")
    else:
        print(f"Copying {tss_src} to {tss_dest}")
        print(f"Copying {frag_len_src} to {frag_len_dest}")
        shutil.copy(tss_src, tss_dest)
        shutil.copy(frag_len_src, frag_len_dest)
    

Copying /zata/zippy/ramirezc/atac-smk/results/tss_enrichment/EA100001-GRCH38.png to /zata/public_html/users/ramirezc/mohd/atac/tss_enrichment/EA100001.png
Copying /zata/zippy/ramirezc/atac-smk/results/frag_len/EA100001-GRCH38.png to /zata/public_html/users/ramirezc/mohd/atac/frag_len/EA100001.png
Copying /zata/zippy/ramirezc/atac-smk/results/tss_enrichment/EA100002-GRCH38.png to /zata/public_html/users/ramirezc/mohd/atac/tss_enrichment/EA100002.png
Copying /zata/zippy/ramirezc/atac-smk/results/frag_len/EA100002-GRCH38.png to /zata/public_html/users/ramirezc/mohd/atac/frag_len/EA100002.png
Copying /zata/zippy/ramirezc/atac-smk/results/tss_enrichment/EA100003-GRCH38.png to /zata/public_html/users/ramirezc/mohd/atac/tss_enrichment/EA100003.png
Copying /zata/zippy/ramirezc/atac-smk/results/frag_len/EA100003-GRCH38.png to /zata/public_html/users/ramirezc/mohd/atac/frag_len/EA100003.png
Copying /zata/zippy/ramirezc/atac-smk/results/tss_enrichment/EA100004-GRCH38.png to /zata/public_html/user